# Tutorial 5, Q3: Icki-Toffee with binary variables

Release after the practical.

**The problem.** The Icki-Toffee selection from Weeks 03–04 (Table 3 of the sheet), now with rules about variety.
$x_i$ is the number of candy $i$ (a whole number) and $\delta_i$ (`xb[i]` in the code) is 1 if candy $i$ is in
the selection. Minimise the cost $\sum_i C_i x_i$ subject to

- weight at least 320 g and at most $1.5 \times 320 = 480$ g: $320 \le \sum_i W_i x_i \le 480$;
- at most 2850 calories: $\sum_i K_i x_i \le 2850$;
- fat at most 45% of the weight: $\sum_i F_i x_i \le 0.45 \sum_i W_i x_i$;
- at least 6 different candies: $\sum_i \delta_i \ge 6$;
- 3 to 6 of each chosen candy, none of the others: $3\,\delta_i \le x_i \le 6\,\delta_i$.

The candies in the code, in table order: 0 Buttery butterscotch, 1 Caramel softie, 2 Orange dream, 3 Coconut Ice, 4 Lemon sorbet, 5 Lime sherbet, 6 Strawberry surprise, 7 Raspberry mousse, 8 Ginger toffee, 9 Rose-water posy, 10 Walnut whirl, 11 Grape bunch.

**How to run it.** Run the first code cell first (it installs Pyomo and the HiGHS solver; the first time takes about a minute), then *Run → Run All Cells*, or run the cells one by one while you read.

**Expected result.** Cost **310**. The browser shows 3 each of Buttery butterscotch, Caramel softie, Lemon sorbet
and Lime sherbet, 6 Rose-water posy and 5 Grape bunch (321 g). The answer sheet has the same first four, 4
Rose-water posy and 6 Grape bunch, also 310. This problem has more than one cheapest selection: any selection that
costs 310 and meets the constraints is correct.

*Original script: `5 Source Files/Python/icki_toffee_cost_binary.py`. Runs in the browser with HiGHS instead of GLPK; the model and the optimal value are unchanged.*

In [ ]:
# Run this cell first.
# In the browser (JupyterLite) it installs the extra packages into this tab;
# in Colab or Anaconda it installs them with pip (already installed = no change).
%pip install pyomo highspy

import sys
if sys.platform == "emscripten":  # only in the browser (JupyterLite): adapt Pyomo to it
    import contextlib, pyomo.common.tee, pyomo.common.formatting as _fmt
    # 1) browsers have no threads: skip Pyomo's threaded capture of solver output
    pyomo.common.tee.capture_output = lambda *args, **kwargs: contextlib.nullcontext()
    for _m in ("pyomo.contrib.appsi.solvers.highs", "pyomo.contrib.solver.solvers.highs"):
        if _m in sys.modules:
            sys.modules[_m].capture_output = pyomo.common.tee.capture_output
    # 2) the notebook's output returns None from write(); model.pprint()/display() need a number
    if not getattr(_fmt.StreamIndenter, "_browser_patched", False):
        class _Counted:
            def __init__(self, s): self._s = s
            def write(self, data): self._s.write(data); return len(data)
            def __getattr__(self, name): return getattr(self._s, name)
        def _init(self, ostream, indent=" " * 4, _orig=_fmt.StreamIndenter.__init__):
            _orig(self, ostream, indent)
            if not isinstance(self.target_os, _Counted):
                object.__setattr__(self, "target_os", _Counted(self.target_os))
        _fmt.StreamIndenter.__init__ = _init
        _fmt.StreamIndenter._browser_patched = True

Load Pyomo's modelling tools.

In [ ]:
from pyomo.environ import *

A concrete model with two variables per candy: how many (`x`, a whole number) and whether it is chosen at all (`xb`, binary).

In [ ]:
# Define integer variables: x[i] = number of candy i (whole number >= 0)
# and binary variables: xb[i] (= delta_i) = 1 if candy i is in the selection
model = ConcreteModel()
model.x = Var(range(12), within=NonNegativeIntegers)
model.xb = Var(range(12), within=Binary)

The data from Table 3, in table order (a `-` in the table is 0 g of fat).

In [ ]:
# Define the objective function values (cost) and the constraint data, candies 0-11 in Table 3 order
cost = [12, 12, 12, 15, 13, 13, 16, 16, 15, 10, 18, 20]  # C_i

fat = [20, 16, 10, 7, 6, 5, 13, 15, 18, 0, 19, 0]  # F_i (g)
cals = [100, 100, 80, 50, 60, 40, 80, 90, 80, 145, 120, 120]  # K_i (calories)
weight = [24, 22, 15, 12, 10, 10, 14, 17, 20, 8, 20, 15]  # W_i (g)

The objective: total cost, minimised.

In [ ]:
# Define the objective function
model.obj = Objective(expr=sum(cost[i] * model.x[i] for i in range(12)), sense=minimize)

The four constraints of Weeks 03–04. The fat constraint has a sum on both sides, which is still linear.

In [ ]:
# Define the inequality constraints
model.weight_min = Constraint(expr=sum(weight[i] * model.x[i] for i in range(12)) >= 320)  # weight >= 320 g
model.weight_max = Constraint(expr=sum(weight[i] * model.x[i] for i in range(12)) <= 1.5 * 320)  # weight <= 480 g
model.cals_max = Constraint(expr=sum(cals[i] * model.x[i] for i in range(12)) <= 2850)  # calories <= 2850
model.fat_weight_min = Constraint(expr=sum(fat[i] * model.x[i] for i in range(12)) <= 0.45 * sum(weight[i] * model.x[i] for i in range(12)))  # fat <= 45% of the weight (an upper limit, despite the name)

Variety: at least 6 of the $\delta_i$ equal 1.

In [ ]:
# Ensure that your selection has at least 6 different candies: sum of delta_i >= 6
model.variety = Constraint(expr=sum(model.xb[i] for i in range(12)) >= 6)

Link the two variables, lower side: if candy $i$ is chosen ($\delta_i = 1$), take at least 3 of it.

In [ ]:
# Ensure your selection includes at least 3 of each of the selected candies.
model.selection_min = ConstraintList()
for i in range(12):
    model.selection_min.add(model.x[i] >= 3 * model.xb[i])  # x_i >= 3 delta_i

Upper side: at most 6 of a chosen candy, and none of a candy that is not chosen ($\delta_i = 0$ gives $x_i \le 0$).

In [ ]:
# Ensure your selection includes at most 6 of each of the selected candies.
model.selection_max = ConstraintList()
for i in range(12):
    model.selection_max.add(model.x[i] <= 6 * model.xb[i])  # x_i <= 6 delta_i (x_i = 0 if not chosen)

Create the HiGHS solver, print the model (`pprint()`: check it against the problem above), solve, and print the results (`display()`).

The HiGHS log appears between the two printouts (in the browser it is printed even without `tee=True`); its *Solving report* should say *Status Optimal*. In `display()`, look at the `Value` column of the variables and at the objective.

In [ ]:
# Create a solver
solver = SolverFactory('appsi_highs')
# Display the model
model.pprint()
# Solve the model
solver.solve(model)
# Display the results
model.display()

## What do you notice?

Without the variety rules the cheapest selection (Week 03 Q2b) costs 268. What do the new rules cost, and which
of the four Week 03–04 constraints actually matter here?

<details><summary>Answer</summary>

Variety costs £42 (310 − 268). The browser's selection weighs 321 g, has 2370 calories and 141 g of fat (44% of
the weight): the 320 g minimum and the fat limit are nearly or exactly reached, while the 480 g cap and the
calorie limit have room to spare. The cheapest boxes are as light as allowed.
</details>